# Feature Engineering

In [2]:
import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, f1_score, make_scorer

In [10]:
PROCESSED_DIR = r"D:\Ameng\Data Science Project\heart-failure-prediction\data\processed"

preprocessor = joblib.load(os.path.join(PROCESSED_DIR, "preprocessor.joblib"))
X_train = pd.read_csv(os.path.join(PROCESSED_DIR, "X_train.csv"))
X_test = pd.read_csv(os.path.join(PROCESSED_DIR, "X_test.csv"))
y_train = pd.read_csv(os.path.join(PROCESSED_DIR, "y_train.csv")).squeeze("columns")
y_test = pd.read_csv(os.path.join(PROCESSED_DIR, "y_test.csv")).squeeze("columns")

In [12]:
print(f"X_train: {X_train.shape}")
print(f"X_test: {X_test.shape}")

X_train: (239, 12)
X_test: (60, 12)


In [19]:
LOG_COLS = ["creatinine_phosphokinase", "serum_creatinine", "platelets", "time"]
SCALE_COLS = ["age", "ejection_fraction", "serum_sodium"]
BIN_COLS = ["anaemia", "diabetes", "high_blood_pressure", "sex", "smoking"]

def to_raw(X_df):
    X = X_df.copy()
    col_map = {f"log__{c}": c for c in LOG_COLS}
    col_map |= {f"scale__{c}": c for c in SCALE_COLS}
    col_map |= {f"pass__{c}": c for c in BIN_COLS}
    X = X.rename(columns=col_map)

    for c in LOG_COLS:
        X[c] = np.expm1(X[c])

    scaler = preprocessor.named_transformers_["scale"]
    for c, mu, sd in zip(SCALE_COLS, scaler.mean_, scaler.scale_):
        X[c] = X[c] * sd + mu
    return X

raw_train, raw_test = to_raw(X_train), to_raw(X_test)
print(raw_train[["age", "serum_creatinine", "sex"]].head())

    age  serum_creatinine  sex
0  58.0               1.0  0.0
1  53.0               0.8  1.0
2  75.0               1.9  1.0
3  64.0               2.4  1.0
4  45.0               1.6  1.0
